In [ ]:
#dataset setup 

import pandas as pd
import numpy as np


df = pd.read_excel("../data/raw/version 1.xlsx")

shape_before = df.shape


df = df.dropna(subset=['Max. Demand at eve. peak (Generation end)'])



df['Actual data of'] = pd.to_datetime(df['Actual data of'], errors='coerce')
df.set_index('Actual data of', inplace=True)
df = df.sort_index() 

C:\Users\Asus\AppData\Local\Temp\ipykernel_11844\3259703427.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Actual data of'] = pd.to_datetime(df['Actual data of'], errors='coerce')


In [ ]:
#Handling missing values 

df_clean = df.copy()

# Impute Temperature using time-based interpolation
df_clean['Maximum Temperature in Dhaka was'] = df_clean['Maximum Temperature in Dhaka was'].interpolate(method='time')

# Impute Regional Demand, Supply, and Load (Missing rows for khulna)

regional_cols = [col for col in df.columns if any(region in col for region in 
                 ['Dhaka', 'Chattogram', 'Rajshahi', 'Mymensingh', 'Sylhet', 'Barishal', 'Rangpur', 'Cumilla', 'Khulna'])]

for col in regional_cols:
    if 'load' in col:
        # For load shedding, forward fill the last known operational state
        df_clean[col] = df_clean[col].ffill()
    else:
        # For demand/supply, use time interpolation
        df_clean[col] = df_clean[col].interpolate(method='time')


print(f"Remaining missing values: {df_clean.isnull().sum().sum()}")

Remaining missing values: 0


In [5]:
# Outlier Detection and Removal on Target Variable using IQR
target_col = 'Max. Demand at eve. peak (Generation end)'
Q1 = df_clean[target_col].quantile(0.25)
Q3 = df_clean[target_col].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Count how many we are dropping
outliers = df_clean[(df_clean[target_col] < lower_bound) | (df_clean[target_col] > upper_bound)]
print(f"Flagged Outliers to Remove: {len(outliers)} days.")

# Filter the dataframe to keep only the normal values
df_clean = df_clean[(df_clean[target_col] >= lower_bound) & (df_clean[target_col] <= upper_bound)]
print(f"Rows remaining after outlier removal: {len(df_clean)}")

Flagged Outliers to Remove: 31 days.
Rows remaining after outlier removal: 1816


In [ ]:
#exporting and saving the preprocessed dataset. 
import os


os.makedirs('../data/processed', exist_ok=True)


df_clean.to_csv("../data/processed/clean_demand_data.csv")


print("--- Preprocessing Summary ---")
print(f"Rows Before: {shape_before[0]} | Rows After: {df_clean.shape[0]}")
print(f"Columns Before: {shape_before[1]} | Columns After: {df_clean.shape[1] + 1}") 

--- Preprocessing Summary ---
Rows Before: 1848 | Rows After: 1816
Columns Before: 41 | Columns After: 41
